In [1]:
import numpy as np
import matplotlib.pyplot as plt

import pyRTC.utils as utils
from pyRTC import *
from astropy.io import fits

import sys
sys.path.append('..')

Spinnaker python SDK installation not found
Spinnaker python SDK installation not found
ximea python SDK installation not found
PI python SDK installation not found
Andor python SDK installation not found


/Users/ellenlee/opt/anaconda3/envs/pyrtc/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
clear_shms(["subApMasks"])

Creating New Shared Memory Object subApMasks
Creating New Shared Memory Object subApMasks_meta
Opening Existing Shared Memory Object subApMasks_meta
Closing subApMasks
Creating New Shared Memory Object subApMasks_gpu_handle
Opening Existing Shared Memory Object subApMasks_gpu_handle_meta
Closing subApMasks_meta
Closing subApMasks_gpu_handle


Traceback (most recent call last):
  File "/Users/ellenlee/opt/anaconda3/envs/pyrtc/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyError: '/subApMasks'
Traceback (most recent call last):
  File "/Users/ellenlee/opt/anaconda3/envs/pyrtc/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyError: '/subApMasks_meta'
Traceback (most recent call last):
  File "/Users/ellenlee/opt/anaconda3/envs/pyrtc/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyError: '/subApMasks_gpu_handle'


In [18]:
shm_names = ["wfs", "wfsRaw", "wfc", "wfc2D", "wfcShape", "signal", "signal2D", "psfShort", "psfLong", "wfsInfo", "loop", "refSlopes", "subApMasks", "cmat", "m2c",
             "simInjectedSlopes"] #list of SHMs to reset
#shm_names = ["slopes", "signal2d"]
# for name in shm_names:
#     try:
#         # Suppress KeyError statements...
#         clear_shms([name])
#     except:
#         pass

clear_shms(shm_names)

Opening Existing Shared Memory Object wfs
Opening Existing Shared Memory Object wfs_meta
Opening Existing Shared Memory Object wfs_meta
Closing wfs
Creating New Shared Memory Object wfs_gpu_handle
Opening Existing Shared Memory Object wfs_gpu_handle_meta
Closing wfs_meta
Opening Existing Shared Memory Object wfsRaw
Opening Existing Shared Memory Object wfsRaw_meta
Closing wfs_gpu_handle
Opening Existing Shared Memory Object wfsRaw_meta
Closing wfsRaw
Creating New Shared Memory Object wfsRaw_gpu_handle
Opening Existing Shared Memory Object wfsRaw_gpu_handle_meta
Closing wfsRaw_meta
Opening Existing Shared Memory Object wfc
Opening Existing Shared Memory Object wfc_meta
Closing wfsRaw_gpu_handle
Opening Existing Shared Memory Object wfc_meta
Closing wfc
Creating New Shared Memory Object wfc_gpu_handle
Opening Existing Shared Memory Object wfc_gpu_handle_meta
Closing wfc_meta
Opening Existing Shared Memory Object wfc2D
Opening Existing Shared Memory Object wfc2D_meta
Closing wfc_gpu_handl

Traceback (most recent call last):
  File "/Users/ellenlee/opt/anaconda3/envs/pyrtc/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyError: '/wfs'
Traceback (most recent call last):
  File "/Users/ellenlee/opt/anaconda3/envs/pyrtc/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyError: '/wfs_meta'
Traceback (most recent call last):
  File "/Users/ellenlee/opt/anaconda3/envs/pyrtc/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyError: '/wfs_gpu_handle'
Traceback (most recent call last):
  File "/Users/ellenlee/opt/anaconda3/envs/pyrtc/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyError: '/wfsRaw'
Traceback (most recent call last):
  File "/Users/ellenlee/opt/anaconda3/envs/pyrtc/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove

### Use ICS

In [22]:
from pyroics import get_ics_proxy

ics = get_ics_proxy()
print(ics.get_available_launchers())

['ImakaDM', 'IRTFASMSimulator', 'FELIXSimulator', 'AndorWFS', 'SlopesProcess', 'Loop', 'ImakaTelemetry']


In [12]:
result = ics.get_config("ao")

In [14]:
result["num_modes"]

7

In [21]:
#ics.launch("ImakaDM")
ics.launch("IRTFASMSimulator")
#ics.launch("AndorWFS")
ics.launch("FELIXSimulator")
ics.launch("SlopesProcess")
ics.launch("Loop")

0

In [26]:
ics.launch("FELIXSimulator")


0

In [4]:
ics.run("wfs", "start")
ics.run("wfc", "start")
ics.run("slopes", "start")
ics.run("loop", "start")

In [19]:
ics.run("slopes", "makeSubApMasks", 0, 0)

3

In [31]:
ics.shutdown_all()

In [2]:
import sys
sys.path.append('..')

from scripts import felix_xy_to_radec, felix_radec_to_xy, find_stars_near_felix

In [3]:
results, err = find_stars_near_felix()

-75.49 260.11
0.0 0.0


In [15]:
ics.is_connected("slopes")

False

In [23]:
ics.run("slopes", "makeSubApMasks", 110, 110)

In [15]:
def slopes_to_modes(slopes):
    ics = get_ics_proxy()

    slopes = ics.run("slopes", "read", False)
    imat = ics.get("loop", "IM")
    modes = np.linalg.pinv(imat) @ slopes
    return modes

In [25]:
modes = slopes_to_modes(slopes)

In [30]:
modes.shape

(7,)

In [12]:
name_brightest = ""
Vmin = 999

for record in results:
    print(record.name)
    print(f"  {record.separation * 206265 :.2f}")
    for i in range(4):
        mag_filter = record.magD[i]
        mag = record.mag[i]
        if mag_filter != "-":
            print(f"  {mag_filter} = {mag:.2f}")
        if mag_filter == "V":
            if mag < Vmin:
                Vmin = mag
                name_brightest = record.name

if Vmin < 999:
    print(name_brightest, '{:.2f}'.format(Vmin))
else:
    print("No guide stars found within a 40\" radius.")

NOMAD1_00109-0006292
  31.89
  V = 15.71
  B = 15.92
  R = 15.59
  J = 14.65
NOMAD1_00109-0006289
  40.76
  V = 15.66
  B = 15.74
  R = 15.49
  J = 14.69
NOMAD1_00109-0006306
  3.41
  R = 19.77
NOMAD1_00109-0006311
  6.93
  B = 20.97
  R = 20.03
NOMAD1_00109-0006308
  13.00
  B = 19.23
  R = 18.99
NOMAD1_00109-0006307
  13.97
  R = 19.48
NOMAD1_00109-0006315
  19.95
  V = 16.94
  B = 17.88
  R = 15.58
  J = 15.33
NOMAD1_00109-0006314
  29.68
  B = 19.70
  R = 19.46
NOMAD1_00109-0006291
  33.26
  V = 17.65
  B = 18.07
  R = 17.27
  J = 15.99
NOMAD1_00109-0006289 15.66


In [70]:
ics.run("slopes", "makeSubApMasks", 97+33, 150-30)

In [38]:
ics.run_and_reset_wfs_shms("wfs", "setBinning", 2)

-1

In [42]:
ics.shutdown_all()

In [24]:
# Step 1: Enter guide star information
ra_guide =  "19_50:46.34"
dec_guide = "+08_54:43.1"
refSlopesFile = f"/home/felix/src/pyrtc/pyRTC-IRTF/IRTF/data/refSlopes/{ra_guide}-{dec_guide}.npy"

In [25]:
print(refSlopesFile)

/home/felix/src/pyrtc/pyRTC-IRTF/IRTF/data/refSlopes/19_50:46.34-+08_54:43.1.npy


In [44]:
refSlopesFile = f"/home/felix/src/pyrtc/pyRTC-IRTF/IRTF/calib/refSlopes/19_50_34.73-+08_53_02.0.npy"

In [49]:
refSlopesFile = f"/home/felix/src/pyrtc/pyRTC-IRTF/IRTF/calib/refSlopes/19_50_35.36-+08_51_34.7.npy"

In [3]:
# Step 2: Set the exposure time of Felix
texpos = 0.98
ics.run("wfs", "setExposure", texpos)

In [7]:
# Step 3: Take darks
tdark = 5
ics.set("wfs", "darkCount", round(tdark / texpos))
ics.run("wfs", "takeDark", False)

In [124]:
# Step 4: Close the loop in imaka RTC

In [134]:
# Step 5.1: Save WFS images
tref = 8
idx_start = 15

ics.run("slopes", "start")
ics.run("loop", "start")
ics.set("tel", "numIter", round(tref / texpos))
ics.run_no_wait("tel", "save", idx_start, True)

In [50]:
# Step 5.2: Record and save ref slopes
#refSlopesFile = f"/home/felix/src/pyrtc/pyRTC-IRTF/IRTF/data/refSlopes/focus_m0.15.npy"

ics.set("slopes", "refSlopesFile", refSlopesFile)
ics.set("slopes", "refSlopeCount", 20)#round(tref / texpos))
ics.run("slopes", "takeRefSlopes")
ics.run("slopes", "saveRefSlopes")

In [48]:
ics.run("slopes", "loadRefSlopes", refSlopesFile)

In [10]:
slopes.run("start")

1

In [9]:
loop.run("start")

1

In [17]:
slopes.setProperty("refSlopeCount", 70)
slopes.run("takeRefSlopes")

1

In [18]:
wfs.shutdown()
dm.shutdown()
slopes.shutdown()
loop.shutdown()
tel.shutdown()

1

In [ ]:
coeffs = np.zeros(7)
amp = 0.1
coeffs[0] = amp
#loop.run("sendToWfc", coeffs)
dm.run("sendToHardware", coeffs)

-1

### Soft RTC

In [2]:
basedir = "/home/felix/src/pyrtc/pyRTC-IRTF/IRTF/config/"
#basedir = "/Users/ellenlee/Documents/pyRTC-IRTF/IRTF/config/"

#conf = utils.read_yaml_file(basedir + "hrtc_wfs_felixsim.yaml")
conf = utils.read_yaml_file("/home/felix/src/pyrtc/pyRTC-IRTF/IRTF/config/hrtc_wfs_andor.yaml")
wfs_conf = conf["wfs"]

conf = utils.read_yaml_file(basedir + "hrtc_slopes.yaml")
slopes_conf = conf["slopes"]

conf = utils.read_yaml_file(basedir + "hrtc_wfcsim.yaml")
#conf = utils.read_yaml_file(basedir + "hrtc_wfc.yaml")
dm_conf = conf["wfc"]

conf = utils.read_yaml_file(basedir + "hrtc_loop.yaml")
loop_conf = conf["loop"]

conf = utils.read_yaml_file(basedir + "hrtc_tel.yaml")
tel_conf = conf["tel"]

#wfs = FELIXSimulator(wfs_conf)
wfs = AndorWFS(wfs_conf)
dm = IRTFASMSimulator(dm_conf)
#dm = ImakaDM(dm_conf)
slopes = SlopesProcess(slopes_conf)
loop = Loop(loop_conf)
tel = ImakaTelemetry(tel_conf)

Opening Existing Shared Memory Object wfsInfo
Thread expose: Priority set to REALTIME
Opening Existing Shared Memory Object wfsInfo_meta
Opening Existing Shared Memory Object wfsRaw
Opening Existing Shared Memory Object wfsRaw_meta
Opening Existing Shared Memory Object wfs
Opening Existing Shared Memory Object wfs_meta
Using HSSpeed index 0: 17.00 MHz
Using VSSpeed index 1: 0.50 µs


Opening Existing Shared Memory Object wfc
Thread sendToHardware: Priority set to REALTIME
Opening Existing Shared Memory Object wfc_meta
Opening Existing Shared Memory Object m2c
Opening Existing Shared Memory Object m2c_meta
Opening Existing Shared Memory Object simInjectedSlopes
Opening Existing Shared Memory Object simInjectedSlopes_meta
Opening Existing Shared Memory Object wfc2D
Opening Existing Shared Memory Object wfc2D_meta
Opening Existing Shared Memory Object wfs_meta
Closing wfs_meta
Thread computeSignal: Priority set to REALTIME
Opening Existing Shared Memory Object wfs
Opening Existing Shared Memory Object wfs_meta
Opening Existing Shared Memory Object wfsInfo_meta
Closing wfsInfo_meta
Opening Existing Shared Memory Object wfsInfo
Opening Existing Shared Memory Object wfsInfo_meta
Opening Existing Shared Memory Object subApMasks
Opening Existing Shared Memory Object subApMasks_meta
Opening Existing Shared Memory Object signal
Opening Existing Shared Memory Object signal_me

In [3]:
wfs.start()
dm.start()
slopes.start()
loop.start()
tel.start()

In [15]:
wfs.stop()
wfs.sdk.AbortAcquisition()
time.sleep(0.5)
wfs.sdk.ShutDown()

del wfs
del slopes
del dm
del loop

20002

In [368]:
from gui.gui_utils import calc_ncpa_lookup, angular_separation

def calc_ncpa_lookup(ra_target, dec_target, ra_guide, dec_guide, ncpa_lookup_file):
    """Calculate NCPA lookup table values.

    Returns
    -------
    ncpa : np.ndarray
        Array of NCPA values for each mode.
    dist_as : float
        Angular separation between target and guide star in arcseconds.
    """
    dist_as = angular_separation(ra_target, dec_target, ra_guide, dec_guide)
    ncpa_lookup = np.load(ncpa_lookup_file)

    n_modes = 7
    ncpa = np.zeros(n_modes, dtype=float)

    for i in range(n_modes):
        a = ncpa_lookup[i,0]
        b = ncpa_lookup[i,1]
        c = ncpa_lookup[i,2]
        #c = 0 # force center to be = 0
        parabola = lambda x: a * (x**2) + b*x +c
        ncpa[i] = parabola(dist_as)

    return ncpa, dist_as
    
ncpa_file = "calib/ncpa_model_7modes.npy"

ra_target = "19:50:47.00"
dec_target = "+08:52:06.0"
ra_guide =  "19:50:34.73"
dec_guide = "+08:53:02.0"
ncpa_coeffs, dist_as = calc_ncpa_lookup(ra_target, dec_target, ra_guide, dec_guide, ncpa_file)
print(dist_as)

190.27327722826166


In [369]:
imat = np.copy(loop.IM)
cmat = np.copy(loop.CM)
imat.shape

(8, 7)

In [370]:
lookupRefSlopes = imat @ ncpa_coeffs
print(ncpa_coeffs)
lookupRefSlopes = np.reshape(lookupRefSlopes, (4,2))

[0.         0.         2.72507062 0.         0.         0.
 0.        ]


In [371]:
lookupRefSlopes

array([[-12.14085602,   9.34673514],
       [-14.75752295,   8.18299342],
       [-13.49016719,  -8.46773421],
       [  7.0845317 ,  17.13974425]])

In [372]:
#oldRefSlopes = slopes.refSlopes.copy()
oldCoeffs = cmat @ oldRefSlopes.flatten()
print(oldRefSlopes)
print(oldCoeffs)

[[-11.050932  11.950879]
 [-12.277639  11.594662]
 [-13.225011 -11.142139]
 [  8.724882  10.259488]]
[ 0.10741949  1.6607544   2.5358696   0.11174741 -0.40609455 -0.14653146
  0.31020325]


In [351]:
testCoeffs = oldCoeffs.copy()
testCoeffs[:2] = 0
testCoeffs[3:] = 0
testCoeffs

array([0.       , 0.       , 2.5358696, 0.       , 0.       , 0.       ,
       0.       ], dtype=float32)

In [373]:
slopes.refSlopes = np.reshape( imat @ testCoeffs , (4,2) )

In [303]:
slopes.refSlopes = lookupRefSlopes

In [378]:
slopes.refSlopes = oldRefSlopes

In [317]:
oldRefSlopes

array([[-10.327399,  12.349455],
       [-13.755118,   9.859696],
       [-13.984613,  -9.358179],
       [  8.177567,  11.799712]], dtype=float32)

In [325]:
slopes.refSlopes

array([[-11.204662 ,   8.6259985],
       [-13.619555 ,   7.5519943],
       [-12.449926 ,  -7.8147783],
       [  6.538236 ,  15.818081 ]], dtype=float32)

In [379]:
dm.flatten()

In [381]:
loop.setGain(0.)
loop.leakyGain = 0.0

shutdown

In [ ]:
wfs.shutdown()
for component in [loop, slopes, dm, wfs, tel]:
    try:
        del component
    except:
        pass

### Debug Andor SDK

In [34]:
from pyAndorSDK2 import atmcd, atmcd_codes, atmcd_errors
sdk = atmcd()
sdk.Initialize("/usr/local/etc/andor/")
sdk.SetAcquisitionMode(atmcd_codes.Acquisition_Mode.SINGLE_SCAN)

20075

In [7]:
sdk.GetNumberVSSpeeds()

(20002, 5)

In [3]:
sdk.ShutDown()

20002

In [47]:
binning = 2
left = 1
top = 1
width = 512
height = 512
size = int( width * height / (binning*binning) )

sdk.SetImage(
        hbin   = binning,
        vbin   = binning,
        hstart = left,
        hend   = left + width - 1,
        vstart = top,
        vend   = top + height - 1
    )

20002

In [54]:
sdk.AbortAcquisition()

20002

In [48]:
sdk.StartAcquisition()

20002

In [49]:
size

65536

In [50]:
ret, raw = sdk.GetMostRecentImage16(size)
raw = np.array(raw, copy=True)

In [52]:
imout = raw.reshape((height//binning, width//binning))

In [53]:
imout.min()


346

In [55]:
sdk.ShutDown()

20002